<a href="https://colab.research.google.com/github/trobinso4124-alt/dflef321/blob/main/video_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import ipywidgets as widgets
from IPython.display import display, clear_output, Video, FileLink
import yt_dlp

print("🚀 Video search ready!")

search_input = widgets.Text(
    placeholder="Type video name or topic...",
    description="🎥 Search:",
    layout=widgets.Layout(width="65%")
)

search_button = widgets.Button(
    description="Find Video",
    button_style="info",
    icon="search"
)

run_button = widgets.Button(
    description="Download & Play",
    button_style="success",
    icon="play",
    disabled=True
)

track_select = widgets.Dropdown(
    options=[],
    description="🔢 Video:",
    disabled=True,
    layout=widgets.Layout(width="65%")
)

output_area = widgets.Output()


def on_search_clicked(b):
    query = search_input.value.strip()
    if not query:
        return

    with output_area:
        clear_output()
        print("📡 Searching...")

        ydl_opts = {
            "quiet": True,
            "no_warnings": True,
            "extract_flat": True
        }

        dropdown_options = []

        try:
            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                info = ydl.extract_info(
                    f"ytsearch20:{query}",
                    download=False
                )

            for idx, entry in enumerate(info.get("entries", [])):
                if not entry:
                    continue

                title = entry.get("title", "Unknown Video")
                video_url = entry.get("webpage_url")

                if not video_url and entry.get("id"):
                    video_url = f"https://www.youtube.com/watch?v={entry['id']}"

                if video_url:
                    label = f"{idx + 1}. {title[:70]}"
                    dropdown_options.append((label, video_url))

            if dropdown_options:
                track_select.options = dropdown_options
                track_select.disabled = False
                run_button.disabled = False
                print("✅ Select a video and click Download & Play.")
            else:
                print("❌ No videos found.")

        except Exception as e:
            print(f"❌ Search error: {e}")


def on_execute_clicked(b):
    video_url = track_select.value

    with output_area:
        clear_output()
        print("📥 Downloading...")

        target_file = "downloaded_video.mp4"

        if os.path.exists(target_file):
            os.remove(target_file)

        ydl_opts = {
            "format": (
                "bestvideo[height<=480][ext=mp4]+"
                "bestaudio[ext=m4a]/"
                "best[height<=480]/best"
            ),
            "outtmpl": "downloaded_video.%(ext)s",
            "merge_output_format": "mp4",
            "noplaylist": True,
            "quiet": True,
            "no_warnings": True
        }

        try:
            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                ydl.download([video_url])

            if os.path.exists(target_file):
                print("🎉 Download complete!")
                display(Video(target_file, embed=True, width=640))
                display(FileLink(target_file))
            else:
                print("❌ The video file was not created.")

        except Exception as e:
            print(f"❌ Download error: {e}")


search_button.on_click(on_search_clicked)
search_input.on_submit(on_search_clicked)
run_button.on_click(on_execute_clicked)

display(widgets.VBox([
    widgets.HBox([search_input, search_button]),
    widgets.HBox([track_select, run_button]),
    output_area
]))